# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [24]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [5]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:12<00:00, 24.01s/it]


In [6]:
len(deals)

30

In [7]:
deals[10].describe()

'Title: Samsung 256GB microSD Express Card for Nintendo Switch 2 for $39 + free shipping\nDetails: Grab this today at its best price. Buy Now at Amazon\nFeatures: \nURL: https://www.dealnews.com/products/Samsung/256-GB-micro-SD-Express-Card-for-Nintendo-Switch-2/497889.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [8]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [9]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [10]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: JLab Talk USB Microphone for $23 + free shipping w/ Prime
Details: It's about $50 or more at most outlets. Buy Now at Amazon
Features: 
URL: https://www.dealnews.com/JLab-Talk-USB-Microphone-for-23-free-shipping-w-Prime/21814768.html?iref=rss-c142

Title: Certified Refurb Eco-Worthy 48V 50Ah LiFePO4 Battery for $320 + free shipping
Details: Use promo code "FRESHFIND20" to drop 

In [11]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Refurbished Samsung Galaxy Z Fold4 with 256GB storage is a high-end foldable Android smartphone featuring a 7.6-inch QXGA+ AMOLED main display with 120Hz adaptive refresh, complemented by a cover display for one-handed use. It’s powered by the Qualcomm Snapdragon 8+ Gen 1 chipset and includes 12GB of RAM for smooth multitasking, plus a versatile triple rear camera system (50MP wide, 12MP ultrawide, 10MP telephoto) for strong photography performance. The model runs Android 12, supports advanced foldable-device features, and comes unlocked for use on multiple carriers.', price=327.0, url='https://www.dealnews.com/products/Samsung/Unlocked-Samsung-Galaxy-Z-Fold-4-256-GB-Android-Smart-Phone/433944.html?iref=rss-c142'), Deal(product_description='Refurbished Apple iPhone 14 Pro with 256GB of internal storage is Apple’s flagship phone from its 14-series, offering a high-quality display and Pro-class performance. The device includes advanced camer

In [12]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Refurbished Samsung Galaxy Z Fold4 with 256GB storage is a high-end foldable Android smartphone featuring a 7.6-inch QXGA+ AMOLED main display with 120Hz adaptive refresh, complemented by a cover display for one-handed use. It’s powered by the Qualcomm Snapdragon 8+ Gen 1 chipset and includes 12GB of RAM for smooth multitasking, plus a versatile triple rear camera system (50MP wide, 12MP ultrawide, 10MP telephoto) for strong photography performance. The model runs Android 12, supports advanced foldable-device features, and comes unlocked for use on multiple carriers.
327.0
https://www.dealnews.com/products/Samsung/Unlocked-Samsung-Galaxy-Z-Fold-4-256-GB-Android-Smart-Phone/433944.html?iref=rss-c142

Refurbished Apple iPhone 14 Pro with 256GB of internal storage is Apple’s flagship phone from its 14-series, offering a high-quality display and Pro-class performance. The device includes advanced cameras, pro-level computational photography, and Face ID security, running iOS with Apple’s e

In [13]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [ ]:
result

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [16]:
load_dotenv(override=True)

True

In [17]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [18]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with a
Pushover token found and starts with u


In [19]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [20]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [29]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [31]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
23:57:23 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic


AuthenticationError: litellm.AuthenticationError: Missing Anthropic API Key - A call is being made to anthropic but no key is set either in the environment variables or via params. Please set `ANTHROPIC_API_KEY` in your environment vars